#1: Imports and Path Setup

In [1]:
import os
import sys
import time
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

In [2]:
PROJECT_ROOT= '/content/drive/MyDrive/Transformer_Mine'
CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, 'checkpoints/toy_overfit')
FIGURES_DIR = os.path.join(PROJECT_ROOT, 'figures')

In [3]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Transformer_Mine

Mounted at /content/drive
/content/drive/MyDrive/Transformer_Mine


In [4]:
from transformer_model import TiedTransformer
from training import (
    label_smoothing_loss,
    TransformerLRSchedule,
    train_step,
    create_checkpoint_manager,
    restore_checkpoint,
)
from data_pipeline import load_raw_pairs, build_tf_dataset
from tokenizer import BPETokenizer



In [5]:
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("TensorFlow:", tf.__version__)
print("Project root:", PROJECT_ROOT)

TensorFlow: 2.20.0
Project root: /content/drive/MyDrive/Transformer_Mine


# 2: What Overfitting Should Look Like

The toy dataset has 20 sentence pairs.
A correct Transformer implementation should be able to memorise 20 examples completely.

## Expected behavior

* Training loss should fall to near zero (< 0.1)
* Greedy decoding of training examples should reproduce the target sentences exactly or near-exactly
* This may take 500 - 3000 steps depending on model size and learning rate

## If overfitting FAILS (loss plateaus above ~1.0)

**STOP. Do not proceed to real training.**

Investigate:

1. Teacher forcing shift (dec_input vs dec_target)
2. Mask correctness (causal mask, padding mask)
3. Loss function (PAD masking, smoothing)
4. Positional encoding (being added correctly)
5. Learning rate (too low = slow, too high = diverges)
6. Weight initialisation
7. Gradient flow (any None gradients)

## Why a small model?

We use a reduced architecture for the toy test.

The purpose is pipeline verification, not paper reproduction.

A smaller model overfits faster and uses less memory.

## COLAB-SCALE CONFIGURATION (toy overfit)

```text
num_layers : 2
d_model    : 128
num_heads  : 4
d_ff       : 512
dropout    : 0.0   (no dropout — we WANT to overfit)
```

## ORIGINAL TRANSFORMER BASE (for reference)

```text
num_layers : 6
d_model    : 512
num_heads  : 8
d_ff       : 2048
dropout    : 0.1
```


# 3: Load Stage A Data and Tokenizer

In [6]:
VOCAB_DIR_A = os.path.join(PROJECT_ROOT, 'vocab/stage_a')
STAGE_A_DIR = os.path.join(PROJECT_ROOT, 'datasets/stage_a')

In [7]:
tokenizer = BPETokenizer.load(VOCAB_DIR_A)

In [8]:
src_lines, tgt_lines = load_raw_pairs(
    os.path.join(STAGE_A_DIR, 'train.en'),
    os.path.join(STAGE_A_DIR, 'train.de'),
)

In [9]:
VOCAB_SIZE = tokenizer.vocab_size
MAX_LEN = 20
BATCH_SIZE = 20

In [10]:
dataset = build_tf_dataset(
    src_lines = src_lines,
    tgt_lines = tgt_lines,
    encode_fn = tokenizer.encode,
    max_len = MAX_LEN,
    batch_size = BATCH_SIZE,
    bos_id = tokenizer.bos_id,
    eos_id = tokenizer.eos_id,
    pad_id = tokenizer.pad_id,
    shuffle = False
)


In [11]:
print(f"Tokenizer vocab size : {VOCAB_SIZE}")
print(f"Sentence pairs : {len(src_lines)}")
print(f"Max sequence length : {MAX_LEN}")
print(f"Batch size : {BATCH_SIZE} (full dataset in one batch)")

Tokenizer vocab size : 137
Sentence pairs : 20
Max sequence length : 20
Batch size : 20 (full dataset in one batch)


In [12]:
for enc_in, dec_in, dec_tgt in dataset.take(1):
    print(f"Batch shapes:")
    print(f"enc_input : {enc_in.shape}")
    print(f"dec_input : {dec_in.shape}")
    print(f"dec_target : {dec_tgt.shape}")
    print()
    print("First example:")
    print(f" src : {tokenizer.decode(enc_in[0].numpy())}")
    print(f" tgt : {tokenizer.decode(dec_tgt[0].numpy())}")

Batch shapes:
enc_input : (20, 20)
dec_input : (20, 20)
dec_target : (20, 20)

First example:
 src : I am a student .
 tgt : Ich bin ein Student .


#4: Build Toy Model

In [33]:
TOY_CONFIG = {
    'num_layers' : 2,
    'd_model' : 128,
    'num_heads' : 4,
    'd_ff' : 512,
    'vocab_size' : VOCAB_SIZE,
    'max_len' : MAX_LEN,
    'dropout' : 0.0,
}

In [34]:
toy_model = TiedTransformer(
    num_layers = TOY_CONFIG['num_layers'],
    d_model = TOY_CONFIG['d_model'],
    num_heads = TOY_CONFIG['num_heads'],
    d_ff = TOY_CONFIG['d_ff'],
    src_vocab = TOY_CONFIG['vocab_size'],
    tgt_vocab = TOY_CONFIG['vocab_size'],
    max_len = TOY_CONFIG['max_len'],
    dropout = TOY_CONFIG['dropout'],
)

In [35]:
toy_opt = tf.keras.optimizers.Adam(
    learning_rate = 3e-4,
    beta_1 = 0.9,
    beta_2 = 0.98,
    epsilon = 1e-9,
)

In [36]:
for enc_in, dec_in, dec_tgt in dataset.take(1):
    _ = toy_model(enc_in, dec_in, training=False)

total_params = sum(tf.size(w).numpy() for w in toy_model.trainable_variables)

print("Toy model built.")
print(f"num_layers : {TOY_CONFIG['num_layers']}")
print(f"d_model : {TOY_CONFIG['d_model']}")
print(f"num_heads : {TOY_CONFIG['num_heads']}")
print(f"d_ff : {TOY_CONFIG['d_ff']}")
print(f"vocab_size : {TOY_CONFIG['vocab_size']}")
print(f"dropout : {TOY_CONFIG['dropout']}")
print(f"parameters : {total_params:,}")
print(f"lr : 3e-4 (fixed — no schedule for toy overfit)")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'tied_transformer_2', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Toy model built.
num_layers : 2
d_model : 128
num_heads : 4
d_ff : 512
vocab_size : 137
dropout : 0.0
parameters : 957,696
lr : 3e-4 (fixed — no schedule for toy overfit)


# 5: Overfit Training Loop

Uses eager mode — @tf.function causes graph tracing issues
in Colab that prevent loss from decreasing on tiny datasets.
 Convergence is measured by prediction accuracy, not loss.
 The label smoothing floor for vocab_size=137 is ~0.49,
 so loss will never reach 0.0 even when predictions are perfect.

In [37]:
def eager_train_step(model, optimizer, src_ids, dec_input, dec_target, vocab_size):
    """
    Single training step in eager mode (no @tf.function).
    Used for toy overfit to avoid Colab graph tracing issues.
    """
    with tf.GradientTape() as tape:
        logits, _ = model(src_ids, dec_input, training=True)
        loss = label_smoothing_loss(
            logits, dec_target, vocab_size, smoothing=0.1
        )
    gradients = tape.gradient(loss, model.trainable_variables)
    gradients, _ = tf.clip_by_global_norm(gradients, clip_norm=1.0)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    return loss

In [38]:
 def check_accuracy(model, dataset, tokenizer, n_total):
    """Check exact match accuracy on training data."""
    enc_in, dec_in, dec_tgt = next(iter(dataset))
    logits, _ = model(enc_in, dec_in, training=False)
    predictions = tf.argmax(logits, axis=-1).numpy()
    n_exact = 0
    for i in range(n_total):
        mask = dec_tgt[i].numpy() != tokenizer.pad_id
        expected = dec_tgt[i].numpy()[mask]
        predicted = predictions[i][mask]
        if (expected == predicted).all():
            n_exact += 1
    return n_exact

In [39]:
MAX_STEPS = 3000
LOG_EVERY = 100
ACC_EVERY = 200   #

losses = []
steps_done = []

In [40]:
dataset_rep = iter(dataset.repeat())

In [41]:

t_start = time.time()

print("Training to overfit 20 sentence pairs")
print("  convergence metric : prediction accuracy ")


for step in range(1, MAX_STEPS + 1):
    enc_in, dec_in, dec_tgt = next(dataset_rep)

    loss = eager_train_step(
        toy_model, toy_opt,
        enc_in, dec_in, dec_tgt,
        VOCAB_SIZE,
    )

    loss_val = loss.numpy()
    losses.append(loss_val)
    steps_done.append(step)

    if step % LOG_EVERY == 0:
        elapsed = time.time() - t_start
        print(f"step {step:>5}  loss={loss_val:.6f}  elapsed={elapsed:.1f}s")

    if step % ACC_EVERY == 0:
        n_exact = check_accuracy(toy_model, dataset, tokenizer, len(src_lines))
        print(f"         accuracy={n_exact}/{len(src_lines)}")
        if n_exact == len(src_lines):
            print(f"\n20/20 exact matches at step {step}.")
            break

elapsed = time.time() - t_start
print(f"\nTraining complete.")
print(f"  Steps run    : {step}")
print(f"  Final loss   : {losses[-1]:.6f}")
print(f"  Time elapsed : {elapsed:.1f}s")

Training to overfit 20 sentence pairs
  convergence metric : prediction accuracy 
step   100  loss=1.071923  elapsed=59.9s
step   200  loss=0.823222  elapsed=115.5s
         accuracy=20/20

20/20 exact matches at step 200.

Training complete.
  Steps run    : 200
  Final loss   : 0.823222
  Time elapsed : 115.6s


#6: Plot Training Loss

#7: Greedy Decoding

In [42]:

enc_in, dec_in, dec_tgt = next(iter(dataset))
logits, _ = toy_model(enc_in, dec_in, training=False)
predictions = tf.argmax(logits, axis=-1).numpy()

n_exact = 0
n_total = len(src_lines)

print("Prediction accuracy on training data")
print("-" * 50)

for i in range(n_total):
    mask      = dec_tgt[i].numpy() != tokenizer.pad_id
    expected  = dec_tgt[i].numpy()[mask]
    predicted = predictions[i][mask]
    exact     = (expected == predicted).all()
    if exact:
        n_exact += 1
    status    = "EXACT" if exact else "diff"
    exp_text  = tokenizer.decode(expected.tolist())
    pred_text = tokenizer.decode(predicted.tolist())
    print(f"[{i+1:>2}] {status}  tgt={exp_text}  pred={pred_text}")

print()
print(f"Exact matches : {n_exact} / {n_total}")
print(f"Accuracy      : {100 * n_exact / n_total:.1f}%")

Prediction accuracy on training data
--------------------------------------------------
[ 1] EXACT  tgt=Ich bin ein Student .  pred=Ich bin ein Student .
[ 2] EXACT  tgt=Sie ist eine Lehrerin .  pred=Sie ist eine Lehrerin .
[ 3] EXACT  tgt=Er ist ein Arzt .  pred=Er ist ein Arzt .
[ 4] EXACT  tgt=Wir sind Freunde .  pred=Wir sind Freunde .
[ 5] EXACT  tgt=Sie sind Studenten .  pred=Sie sind Studenten .
[ 6] EXACT  tgt=Ich mag Musik .  pred=Ich mag Musik .
[ 7] EXACT  tgt=Sie mag Bücher .  pred=Sie mag Bücher .
[ 8] EXACT  tgt=Er mag Sport .  pred=Er mag Sport .
[ 9] EXACT  tgt=Ich gehe zur Schule .  pred=Ich gehe zur Schule .
[10] EXACT  tgt=Sie geht zur Arbeit .  pred=Sie geht zur Arbeit .
[11] EXACT  tgt=Er geht nach Hause .  pred=Er geht nach Hause .
[12] EXACT  tgt=Ich habe einen Hund .  pred=Ich habe einen Hund .
[13] EXACT  tgt=Sie hat eine Katze .  pred=Sie hat eine Katze .
[14] EXACT  tgt=Wir haben ein Auto .  pred=Wir haben ein Auto .
[15] EXACT  tgt=Ich bin glücklich .  pred=

In [44]:
# ============================================================
# PHASE 12 - CELL 9: Save Checkpoint and Log Results
# ============================================================

import csv
import json

# Save checkpoint
ckpt_toy, manager_toy = create_checkpoint_manager(
    toy_model, toy_opt, CHECKPOINT_DIR, max_to_keep=1
)
save_path = manager_toy.save()
print(f"Overfit checkpoint saved: {save_path}")

# Log to results CSV
RESULTS_CSV = os.path.join(PROJECT_ROOT, 'experiments/results.csv')

record = {
    'experiment_id'  : 'toy_overfit_01',
    'model_config'   : 'COLAB-SCALE (2L, d128, h4, ff512)',
    'dataset'        : 'Stage A — 20 handmade EN-DE pairs',
    'dataset_size'   : 20,
    'epochs'         : 'N/A (step-based)',
    'steps'          : step,
    'batch_size'     : BATCH_SIZE,
    'd_model'        : TOY_CONFIG['d_model'],
    'num_heads'      : TOY_CONFIG['num_heads'],
    'num_layers'     : TOY_CONFIG['num_layers'],
    'd_ff'           : TOY_CONFIG['d_ff'],
    'dropout'        : TOY_CONFIG['dropout'],
    'label_smoothing': 0.1,
    'learning_rate'  : 'TransformerLRSchedule(d_model=128, warmup=200)',
    'warmup_steps'   : 200,
    'parameters'     : total_params,
    'training_time'  : f'{elapsed:.1f}s',
    'validation_loss': f'{losses[-1]:.6f}',
    'BLEU'           : 'N/A',
    'checkpoint'     : save_path,
    'notes'          : f'Toy overfit. Final loss={losses[-1]:.6f}. '
                       f'Exact matches={n_exact}/{n_total}',
}

FIELDNAMES = [
    'experiment_id', 'model_config', 'dataset', 'dataset_size',
    'epochs', 'steps', 'batch_size', 'd_model', 'num_heads',
    'num_layers', 'd_ff', 'dropout', 'label_smoothing',
    'learning_rate', 'warmup_steps', 'parameters', 'gpu',
    'training_time', 'validation_loss', 'BLEU', 'checkpoint', 'notes',
]

file_exists = os.path.exists(RESULTS_CSV)
with open(RESULTS_CSV, 'a', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
    if not file_exists:
        writer.writeheader()
    row = {k: record.get(k, '') for k in FIELDNAMES}
    writer.writerow(row)

print(f"Results logged to: {RESULTS_CSV}")
print()
print("Summary:")
print(f"  Steps         : {step}")
print(f"  Final loss    : {losses[-1]:.6f}")
print(f"  Exact matches : {n_exact}/{n_total}")
print(f"  Time          : {elapsed:.1f}s")

Overfit checkpoint saved: /content/drive/MyDrive/Transformer_Mine/checkpoints/toy_overfit/ckpt-1
Results logged to: /content/drive/MyDrive/Transformer_Mine/experiments/results.csv

Summary:
  Steps         : 200
  Final loss    : 0.823222
  Exact matches : 20/20
  Time          : 115.6s
